> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 06 · LLM-as-a-judge: grade your CampusMate

**CampusMate v3 (an agent with a report card)**  ·  ⏱️ Time budget: **15 min theory + 30 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

Your CampusMate answers questions. But are the answers **right**? You can't read a thousand answers yourself every day.
**LLM-as-a-judge** means: a second model reads each answer, compares it with the correct facts, and gives it a mark with a reason.
That gives your agent a report card you can check after every change.

### What you'll do
- Build a judge with a marking scheme (1 to 5)
- Warm up: let it mark one good and one bad answer
- Write a small exam with the right answers taken from the campus data
- Ask **your** CampusMate the exam, and get its report card

## Settings

In [ ]:
# ---- Step 0: make sure the workshop libraries are in THIS notebook's Python (macOS, Windows, Linux) ----
import sys, subprocess, importlib, importlib.util
if sys.version_info < (3, 10):
    raise SystemExit(f"This notebook runs Python {sys.version.split()[0]}. Choose the .venv kernel (Python 3.10+) and run again.")
if any(importlib.util.find_spec(m) is None for m in ("strands", "strands_tools", "bedrock_agentcore", "mcp_proxy_for_aws", "boto3", "dotenv")):
    print("Installing the workshop libraries into", sys.executable, "(1-2 minutes, only the first time)...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                               "strands-agents==1.64.0", "strands-agents-tools==0.9.0", "bedrock-agentcore==1.25.4",
                               "mcp-proxy-for-aws==1.8.0", "boto3==1.43.126", "botocore==1.43.126",
                               "flask==3.1.3", "python-dotenv==1.2.4", "requests==2.34.2"])
        importlib.invalidate_caches()
    except subprocess.CalledProcessError:
        raise SystemExit("Could not install here. Choose the .venv kernel (VS Code: Select Kernel -> .venv; "
                         "Jupyter: start it from a terminal after activating .venv), then run this cell again.")
print("Libraries ready in:", sys.executable)

# Settings for this lab
import json, uuid, pathlib

REGION = "us-east-1"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"              # Amazon Nova 2 Lite
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"
print("Region:", REGION, "| Model:", MODEL_ID)

## Your CampusMate (from notebook 04)

The same two tools and system prompt you built in notebook 04. Each question gets a **brand-new agent**,
so every answer is marked on its own (like a fair exam: no help from earlier questions).

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel


@tool
def get_course_info(query: str) -> str:
    """Look up AIDD courses by course code (e.g. AIDD303) or a keyword in the title/description (e.g. docker).

    Args:
        query: A course code or a keyword.
    """
    q = query.lower()
    courses = json.loads((DATA_DIR / "courses.json").read_text(encoding="utf-8"))["courses"]
    hits = [c for c in courses if q == c["code"].lower() or q in (c["title"] + " " + c["description"]).lower()]
    return json.dumps(hits[:5]) if hits else json.dumps({"found": False, "codes": [c["code"] for c in courses]})


@tool
def get_campus_policy(topic: str) -> str:
    """Return CampusMate rules for a topic: attendance, internal_assessments, semester_end_exams, fees, leave,
    re_evaluation, library, hackathons. Always use this instead of guessing a rule.

    Args:
        topic: one of the topics above
    """
    policies = json.loads((DATA_DIR / "campus_policies.json").read_text(encoding="utf-8"))["policies"]
    return json.dumps(policies.get(topic.lower().strip(), {"found": False, "topics": list(policies)}))


def campusmate(question):
    """Ask a fresh CampusMate one question and return the answer text."""
    agent = Agent(model=BedrockModel(model_id=MODEL_ID, max_tokens=1200), tools=[get_course_info, get_campus_policy],
                  system_prompt="You are CampusMate for AIDD students. Use tools for courses and rules; never invent rules. Be brief.",
                  callback_handler=None)
    return str(agent(question)).strip()

print(campusmate("In one line: what can you help me with?"))

## What is LLM-as-a-judge?

| | |
|---|---|
| **Student** | CampusMate writes an answer |
| **Answer key** | the correct facts (from our campus data) |
| **Examiner** | a second model, the **judge**, compares the two |
| **Marks** | a score from 1 to 5 and one line of reason |

🏫 **Analogy:** an examiner with a marking scheme. They don't need to know the subject by heart; they check your answer against the key.

Two rules make a judge fair:
1. Give it the **right answer** to compare with. Never ask "is this good?" without a key.
2. Set **temperature to 0**, so the same answer always gets the same mark.

## Step 1: build the judge and its marking scheme

| Score | Meaning |
|---|---|
| 5 | Fully correct and helpful |
| 4 | Correct, small detail missing |
| 3 | Partly correct |
| 2 | Mostly wrong |
| 1 | Wrong, or made up |

We ask the judge to reply in a fixed shape (`Verdict`): a score, correct yes/no, and a reason. Strands fills it in for us.

In [ ]:
from pydantic import BaseModel, Field
from strands import Agent
from strands.models import BedrockModel

class Verdict(BaseModel):
    score: int = Field(description="1 = wrong or made up, 3 = partly correct, 5 = fully correct and helpful")
    correct: bool = Field(description="True if the answer agrees with the reference facts")
    reason: str = Field(description="one short sentence explaining the score")

JUDGE_PROMPT = """You are a strict, fair examiner.
Grade the ANSWER only against the REFERENCE facts. Do not use your own knowledge of the college.
If the question is off-topic and the reference says the assistant should decline, a polite refusal is fully correct."""

judge_model = BedrockModel(model_id=MODEL_ID, temperature=0, max_tokens=1000, streaming=False)

def judge(question, reference, answer):
    """Return a Verdict for one answer."""
    examiner = Agent(model=judge_model, system_prompt=JUDGE_PROMPT, callback_handler=None)
    prompt = f"QUESTION: {question}\nREFERENCE: {reference}\nANSWER: {answer}"
    return examiner(prompt, structured_output_model=Verdict).structured_output

print("Judge ready:", MODEL_ID)

## Step 2: warm-up, one good answer and one bad answer

Same question, two answers we wrote by hand. The judge should give the good one a high mark and the bad one a low mark.

In [ ]:
question = "What is the minimum attendance to write the semester-end exam?"
reference = "75% in each course. 65-74% may be condoned once per semester with a medical certificate (HOD approval)."

good = "You need at least 75% in each course. Between 65% and 74% you can ask for condonation once a semester with a medical certificate."
bad = "You need 60% attendance overall."

for label, answer in [("good answer", good), ("bad answer", bad)]:
    v = judge(question, reference, answer)
    print(f"{label:12s} -> score {v.score}/5 | correct={v.correct} | {v.reason}")

## Step 3: a small exam, with the answer key from the campus data

The right answers come straight from `data/campus_policies.json` and `data/courses.json`, so the judge compares against the real (fictional) rules.
The last question is off-topic: CampusMate should politely decline.

In [ ]:
policies = json.loads((DATA_DIR / "campus_policies.json").read_text(encoding="utf-8"))["policies"]
courses = {c["code"]: c for c in json.loads((DATA_DIR / "courses.json").read_text(encoding="utf-8"))["courses"]}
docker = courses["AIDD301"]

EXAM = [
    ("What is the minimum attendance to write the semester-end exam?", " ".join(policies["attendance"]["rules"][:3])),
    ("What is the library fine per day?", " ".join(policies["library"]["rules"])),
    ("How is the Docker course assessed?", f"{docker['code']} {docker['title']}: assessment {docker['assessment']}"),
    ("Can I take part in an external hackathon during the semester?", " ".join(policies["hackathons"]["rules"][:2])),
    ("Who will win the IPL this year?", "Off-topic. CampusMate should politely decline and say what it can help with."),
]
for q, ref in EXAM:
    print("Q:", q, "\n   key:", ref[:110], "\n")

## Step 4: ask your CampusMate the exam, and mark every answer

This calls CampusMate 5 times and the judge 5 times (about a minute).

In [ ]:
report = []
for q, ref in EXAM:
    answer = campusmate(q)
    v = judge(q, ref, answer)
    report.append({"question": q, "score": v.score, "correct": v.correct, "reason": v.reason, "answer": answer})
    print(f"{v.score}/5 {'✅' if v.correct else '❌'}  {q}\n      judge: {v.reason}\n")

average = sum(r["score"] for r in report) / len(report)
passed = sum(r["correct"] for r in report)
print(f"REPORT CARD: average {average:.1f}/5 | {passed}/{len(report)} correct")

## Step 5: read the report card

- **A low score?** Read the judge's reason, then CampusMate's full answer (`report[0]["answer"]`, `report[1]["answer"]`, ...).
  Usually the fix is a clearer tool docstring or system prompt (notebook 04), not a bigger model.
- **Judges can be wrong too.** Check a few marks by hand, especially the low ones.
- **Run the same exam after every change.** If the average drops, your change made CampusMate worse. That's how teams ship agents safely.
- Judging costs tokens too: here about 10 model calls, well under one rupee.

In [ ]:
# Look at one full answer (change the number to see others: 0 to 4)
print(report[0]["question"], "\n")
print(report[0]["answer"])

## ✍️ Your turn

Add your own exam question with its answer key (pick any rule from `data/campus_policies.json`, for example leave or fees), then mark CampusMate's answer.

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
MY_QUESTION = "TODO"     # TODO: e.g. "How many days of leave can CampusMate file for me?"
MY_REFERENCE = "TODO"    # TODO: e.g. " ".join(policies["leave"]["rules"])

if "TODO" not in (MY_QUESTION, MY_REFERENCE):
    answer = campusmate(MY_QUESTION)
    v = judge(MY_QUESTION, MY_REFERENCE, answer)
    print("CampusMate:", answer, "\n")
    print(f"Judge: {v.score}/5 | correct={v.correct} | {v.reason}")
else:
    print("Skipped: fill in MY_QUESTION and MY_REFERENCE first.")

## ✅ Checkpoint

You are on track if:
- Step 2: the good answer scored 4-5 and the bad answer scored 1-2
- Step 4 printed a report card with an average score
- You can explain one judge reason in your own words

Next, notebook 07: CampusMate (with memory) moves to the cloud.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'strands'` | The notebook isn't using `.venv` | Run the Settings cell again, or VS Code: Select Kernel → `.venv` |
| `ThrottlingException` | Many students calling the model at once | Wait 30 seconds and run the cell again |
| `ValidationException ... structured output` | The judge reply couldn't be read | Run the cell again (temperature 0 makes this rare) |